In [ ]:
#-- load libraries
library(readr)
library(dplyr)
library(ggplot2)
library(tidyr)
library(forcats)
library(paletteer)
library(ggalluvial)
library(scales)
library(lubridate)
library(cowplot)
library(readxl)

In [ ]:
#-- Read in outcome data
outcome <- read_delim("/home/jupyter/workspaces/geneticsofantidepressantresponse/data/Main_Outcomes_90days_noBIP_noPsychotic.csv", delim = ",", show_col_types = FALSE)
combo <- read.csv("/home/jupyter/workspaces/geneticsofantidepressantresponse/data/Combination_Therapy_noBIP_noPsychotic.csv") %>% select(person_id, Case)
aug <- read.csv("/home/jupyter/workspaces/geneticsofantidepressantresponse/data/Augmentation_Therapy_noBIP_noPsychotic.csv") %>% select(person_id, Case)

#-- Load Phenotypes
pheno <- read.csv("/home/jupyter/workspaces/geneticsofantidepressantresponse/data/Phenotypes.csv")
first_dispense <- read.csv("/home/jupyter/workspaces/geneticsofantidepressantresponse/data/First_Dispense.csv")
pheno <- pheno %>%
    left_join(first_dispense, by = "person_id")

#-- Join outcomes with phenotypes
both <- outcome %>%
  left_join(pheno, by = "person_id") %>%
  mutate(
    age_at_first_AD = interval(date_of_birth, DateFirstDispense) %/% years(1)
  )

#-- Pooled switching case and control counts
switch <- outcome %>%
    filter(FinalClassification.treatment == "Switching")
case <- length(unique(switch$person_id))

In [ ]:
#-- Summarise the number of AD scripts per person
PrescriptionData <- read.csv("/home/jupyter/workspaces/geneticsofantidepressantresponse/data/AD_processed.csv") %>%
  mutate(drug_exposure_start_date = as.Date(drug_exposure_start_date),
         final_end_date = as.Date(final_end_date))

PrescriptionData_AD <- PrescriptionData %>%
  filter(!Category %in% c("Atypical Antipsychotic", "Mood_Stabilizer", "Typical Antipsychotic", 
                          "Serotonin_Precursor", "AminoAcid", "Augmentation"))

incl_pres <- PrescriptionData_AD %>%
    filter(person_id %in% outcome$person_id)

num_pres <- incl_pres %>%
    group_by(person_id) %>%
    summarise(
    total = n())

In [ ]:
num_pres_plot <- num_pres %>%
  mutate(total_capped = ifelse(total > 250, 251, total))

px <- ggplot(num_pres_plot, aes(x = total_capped)) +
  geom_histogram(binwidth = 10, fill = "steelblue", color = "white") +
  scale_x_continuous(
    breaks = c(0, 50, 100, 150, 200, 250),
    labels = c("0", "50", "100", "150", "200", "250+")
  ) +
  labs(x = "Total AD prescriptions", y = "Number of people in analytical sample") +
  theme_classic()

num_pres_plot_zoom <- num_pres %>%
  filter(total <= 10)

pxx <- ggplot(num_pres_plot_zoom, aes(x = total)) +
  geom_histogram(binwidth = 1, fill = "steelblue", color = "white") +
  scale_x_continuous(
    breaks = c(0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10),
    labels = c("0", "1", "2", "3", "4", "5", "6", "7", "8", "9", "10")
  ) +
  labs(x = "Total AD prescriptions", y = "Number of people in analytical sample") +
  theme_classic()

all <- plot_grid(px, pxx, nrow = 2, labels = c("A", "B"))
all

ggsave(
  filename = "/home/jupyter/workspaces/geneticsofantidepressantresponse/data/Figures/Supp_Num_Prescriptions.png",
  plot     = all,
  width    = 6,
  height   = 8,
  dpi      = 600
)

In [ ]:
#==== Summarise group counts ===============================================================
outcome = outcome %>%
    mutate(
        drug.treatment = case_when(
            drug.treatment == "Serotonin_Modulator:vortioxetine" ~ "SM:vortioxetine",
            drug.treatment == "Serotonin_Modulator:vilazodone" ~ "SM:vilazadone",
            drug.treatment == "NMDA_Antagonist:esketamine" ~ "NMDAA:esketamine",
            TRUE ~ drug.treatment))

C <- outcome %>%
  filter(FinalClassification.treatment == "Continuation")
C_clean <- C %>%
    filter(!person_id %in% aug$person_id[aug$Case == 1]) %>%
    filter(!person_id %in% combo$person_id[combo$Case == 1])
D_S <- outcome %>%
    filter(FinalClassification.treatment != "Continuation")
final <- rbind(C_clean, D_S)

order <- final %>%
  group_by(drug.treatment) %>%
  summarize(NumPeople = n(), .groups = 'drop') %>%
  as.data.frame() %>%
  arrange(desc(NumPeople)) %>%
  slice_head(n = 15)

order <- order$drug.treatment

group_counts <- final %>%
  mutate(drug.treatment = ifelse(!drug.treatment %in% order, "Other", drug.treatment)) %>%
  group_by(person_id, drug.treatment, FinalClassification.treatment) %>%
  ungroup() %>%
  group_by(drug.treatment, FinalClassification.treatment) %>%
  summarize(NumPeople = n(), .groups = 'drop') %>%
  arrange(drug.treatment) %>%
  as.data.frame()

group_counts$drug.treatment = factor(group_counts$drug.treatment, levels = c(order, "Other"))
group_counts$FinalClassification = factor(group_counts$FinalClassification, levels = c("Switching", "Discontinuation", "Continuation"))

colors <- c(
    "Discontinuation" = "#0D47A1FF",
    "Switching" = "#F57F17FF",
    "Continuation" = "#50c878")

p1 <- ggplot(group_counts, aes(x = drug.treatment, y = NumPeople, fill = FinalClassification)) +
  geom_bar(position = "stack", stat = "identity") +
  theme_classic() +
  scale_fill_manual(values = colors, name = "Outcome") +
  xlab("Common antidepressants") +
  ylab("AoU Participant Count") +
  theme(
    axis.text.x = element_text(angle = 60, hjust = 1, color = "black"),
    axis.title = element_text(face="bold"),
    legend.position = c(0.85, 0.85),
    legend.justification = c(1, 1),
    legend.background = element_rect(fill = "white", color = NA),
    legend.key.size = unit(0.4, "cm"),
    legend.title = element_text(face = "bold")
  ) +
  scale_y_continuous(
    breaks = seq(0, 30000, by = 5000),
    labels = function(x) paste0(x / 1000, "K"))

In [ ]:
dat <- read_excel("/home/jupyter/workspaces/geneticsofantidepressantresponse/data/All_Results_PGS_LL.xlsx", sheet = "Table9")
dat_full <- dat %>%
    filter(Cohort == "Full")

order <- dat_full %>%
    filter(DrugName.treatment != "Other") %>%
    group_by(DrugName.treatment) %>%
    summarise(Total = sum(NumPeople)) %>%
    arrange(desc(Total)) %>%
    pull(DrugName.treatment)

dat_full$FinalClassification.treatment = factor(dat_full$FinalClassification.treatment, levels = c("Switching", "Discontinuation", "Continuation"))
dat_full$DrugName.treatment = factor(dat_full$DrugName.treatment, levels = c(order, "Other"))


colors <- c(
    "Discontinuation" = "#0D47A1FF",
    "Switching" = "#F57F17FF",
    "Continuation" = "#50c878")

p1_ll <- ggplot(dat_full, aes(x = DrugName.treatment, y = NumPeople, fill = FinalClassification.treatment)) +
  geom_bar(position = "stack", stat = "identity") +
  theme_classic() +
  scale_fill_manual(values = colors, name = "Outcome") +
  xlab("Common antidepressant") +
  ylab("PharmLines Participant Count") +
  theme(
    axis.text.x = element_text(angle = 60, hjust = 1, color = "black"),
    axis.title = element_text(face="bold"),
    legend.position = c(0.85, 0.85),
    legend.justification = c(1, 1),
    legend.background = element_rect(fill = "white", color = NA),
    legend.key.size = unit(0.4, "cm"),
    legend.title = element_text(face = "bold")
  ) +
  scale_y_continuous(
    breaks = seq(0, 4500, by = 1000),
    labels = function(x) paste0(x / 1000, "K")) 

In [ ]:
top <- plot_grid(p1, p1_ll, ncol = 2, labels = c("A", "B"))
top

In [ ]:
#==== Summarise group counts within MDD ===============================================================
mdd = both %>%
    filter(Depression == 1) %>%
    mutate(
        drug.treatment = case_when(
            drug.treatment == "Serotonin_Modulator:vortioxetine" ~ "SM:vortioxetine",
            drug.treatment == "Serotonin_Modulator:vilazodone" ~ "SM:vilazadone",
            drug.treatment == "NMDA_Antagonist:esketamine" ~ "NMDAA:esketamine",
            TRUE ~ drug.treatment))

C <- mdd %>%
  filter(FinalClassification.treatment == "Continuation")
C_clean <- C %>%
    filter(!person_id %in% aug$person_id[aug$Case == 1]) %>%
    filter(!person_id %in% combo$person_id[combo$Case == 1])
D_S <- mdd %>%
    filter(FinalClassification.treatment != "Continuation")
final <- rbind(C_clean, D_S)

order <- final %>%
  group_by(drug.treatment) %>%
  summarize(NumPeople = n(), .groups = 'drop') %>%
  as.data.frame() %>%
  arrange(desc(NumPeople)) %>%
  slice_head(n = 15)

order <- order$drug.treatment

group_counts <- final %>%
  mutate(drug.treatment = ifelse(!drug.treatment %in% order, "Other", drug.treatment)) %>%
  group_by(person_id, drug.treatment, FinalClassification.treatment) %>%
  ungroup() %>%
  group_by(drug.treatment, FinalClassification.treatment) %>%
  summarize(NumPeople = n(), .groups = 'drop') %>%
  arrange(drug.treatment) %>%
  as.data.frame()


group_counts$drug.treatment = factor(group_counts$drug.treatment, levels = c(order, "Other"))
group_counts$FinalClassification = factor(group_counts$FinalClassification, levels = c("Switching", "Discontinuation", "Continuation"))

colors <- c(
    "Discontinuation" = "#0D47A1FF",
    "Switching" = "#F57F17FF",
    "Continuation" = "#50c878")

p1b <- ggplot(group_counts, aes(x = drug.treatment, y = NumPeople, fill = FinalClassification)) +
  geom_bar(position = "stack", stat = "identity") +
  theme_classic() +
  scale_fill_manual(values = colors, name = "Outcome") +
  xlab("Common antidepressants") +
  ylab("AoU Participant Count") +
  labs(fill = "Antidepressant\nOutcome", title = "AoU: Recorded Depression") +
  theme(
    axis.text.x = element_text(angle = 60, hjust = 1, color = "black"),
    axis.title = element_text(face="bold"),
    legend.position = "none"
  ) +
  scale_y_continuous(
    breaks = seq(0, 20000, by = 3000),
    labels = function(x) paste0(x / 1000, "K"))
p1b

In [ ]:
#==== Summarise group counts with no MDD ===============================================================
nmdd = both %>%
    filter(Depression == 0) %>%
    mutate(
        drug.treatment = case_when(
            drug.treatment == "Serotonin_Modulator:vortioxetine" ~ "SM:vortioxetine",
            drug.treatment == "Serotonin_Modulator:vilazodone" ~ "SM:vilazadone",
            drug.treatment == "NMDA_Antagonist:esketamine" ~ "NMDAA:esketamine",
            TRUE ~ drug.treatment))

C <- nmdd %>%
  filter(FinalClassification.treatment == "Continuation")
C_clean <- C %>%
    filter(!person_id %in% aug$person_id[aug$Case == 1]) %>%
    filter(!person_id %in% combo$person_id[combo$Case == 1])
D_S <- nmdd %>%
    filter(FinalClassification.treatment != "Continuation")
final <- rbind(C_clean, D_S)

#order <- final %>%
#  group_by(drug.treatment) %>%
#  summarize(NumPeople = n(), .groups = 'drop') %>%
#  as.data.frame() %>%
#  arrange(desc(NumPeople)) %>%
#  slice_head(n = 15)

#order <- order$drug.treatment

group_counts <- final %>%
  mutate(drug.treatment = ifelse(!drug.treatment %in% order, "Other", drug.treatment)) %>%
  group_by(person_id, drug.treatment, FinalClassification.treatment) %>%
  ungroup() %>%
  group_by(drug.treatment, FinalClassification.treatment) %>%
  summarize(NumPeople = n(), .groups = 'drop') %>%
  arrange(drug.treatment) %>%
  as.data.frame()


group_counts$drug.treatment = factor(group_counts$drug.treatment, levels = c(order, "Other"))
group_counts$FinalClassification = factor(group_counts$FinalClassification, levels = c("Switching", "Discontinuation", "Continuation"))

colors <- c(
    "Discontinuation" = "#0D47A1FF",
    "Switching" = "#F57F17FF",
    "Continuation" = "#50c878")

p1c <- ggplot(group_counts, aes(x = drug.treatment, y = NumPeople, fill = FinalClassification)) +
  geom_bar(position = "stack", stat = "identity") +
  theme_classic() +
  scale_fill_manual(values = colors, name = "Outcome") +
  xlab("Common antidepressants") +
  ylab("AoU Participant Count") +
  labs(fill = "Antidepressant\nOutcome", title = "AoU: No Recorded Depression") +
  theme(
    axis.text.x = element_text(angle = 60, hjust = 1, color = "black"),
    axis.title = element_text(face="bold"),
    legend.position = c(0.85, 0.85),
    legend.justification = c(1, 1),
    legend.background = element_rect(fill = "white", color = NA),
    legend.key.size = unit(0.4, "cm"),
    legend.title = element_text(face = "bold")
  ) +
  scale_y_continuous(
    breaks = seq(0, 10000, by = 2000),
    labels = function(x) paste0(x / 1000, "K"))
p1c

In [ ]:
#==== Summarise group counts within MDD ===============================================================
cidi_mdd = both %>%
    filter(MDD_Incl == 1) %>%
    mutate(
        drug.treatment = case_when(
            drug.treatment == "Serotonin_Modulator:vortioxetine" ~ "SM:vortioxetine",
            drug.treatment == "Serotonin_Modulator:vilazodone" ~ "SM:vilazadone",
            drug.treatment == "NMDA_Antagonist:esketamine" ~ "NMDAA:esketamine",
            TRUE ~ drug.treatment))

C <- cidi_mdd %>%
  filter(FinalClassification.treatment == "Continuation")
C_clean <- C %>%
    filter(!person_id %in% aug$person_id[aug$Case == 1]) %>%
    filter(!person_id %in% combo$person_id[combo$Case == 1])
D_S <- cidi_mdd %>%
    filter(FinalClassification.treatment != "Continuation")
final <- rbind(C_clean, D_S)

#order <- final %>%
#  group_by(drug.treatment) %>%
#  summarize(NumPeople = n(), .groups = 'drop') %>%
#  as.data.frame() %>%
#  arrange(desc(NumPeople)) %>%
#  slice_head(n = 15)

group_counts <- final %>%
  mutate(drug.treatment = ifelse(!drug.treatment %in% order, "Other", drug.treatment)) %>%
  group_by(person_id, drug.treatment, FinalClassification.treatment) %>%
  ungroup() %>%
  group_by(drug.treatment, FinalClassification.treatment) %>%
  summarize(NumPeople = n(), .groups = 'drop') %>%
  arrange(drug.treatment) %>%
  as.data.frame()


group_counts$drug.treatment = factor(group_counts$drug.treatment, levels = c(order, "Other"))
group_counts$FinalClassification = factor(group_counts$FinalClassification, levels = c("Switching", "Discontinuation", "Continuation"))

colors <- c(
    "Discontinuation" = "#0D47A1FF",
    "Switching" = "#F57F17FF",
    "Continuation" = "#50c878")

p1d <- ggplot(group_counts, aes(x = drug.treatment, y = NumPeople, fill = FinalClassification)) +
  geom_bar(position = "stack", stat = "identity") +
  theme_classic() +
  scale_fill_manual(values = colors, name = "Outcome") +
  xlab("Common antidepressants") +
  ylab("AoU Participant Count") +
  labs(fill = "Antidepressant\nOutcome", title = "AoU: CIDI-MDD") +
  theme(
    axis.text.x = element_text(angle = 60, hjust = 1, color = "black"),
    axis.title = element_text(face="bold"),
    legend.position = "none"
  ) +
  scale_y_continuous(
    breaks = seq(0, 50000, by = 200),
    labels = function(x) paste0(x / 1000, "K"))
p1d

In [ ]:
#==== Summarise group counts within MDD ===============================================================
cidi_nomdd = both %>%
    filter(MDD_Incl == 0) %>%
    mutate(
        drug.treatment = case_when(
            drug.treatment == "Serotonin_Modulator:vortioxetine" ~ "SM:vortioxetine",
            drug.treatment == "Serotonin_Modulator:vilazodone" ~ "SM:vilazadone",
            drug.treatment == "NMDA_Antagonist:esketamine" ~ "NMDAA:esketamine",
            TRUE ~ drug.treatment))

C <- cidi_nomdd %>%
  filter(FinalClassification.treatment == "Continuation")
C_clean <- C %>%
    filter(!person_id %in% aug$person_id[aug$Case == 1]) %>%
    filter(!person_id %in% combo$person_id[combo$Case == 1])
D_S <- cidi_nomdd %>%
    filter(FinalClassification.treatment != "Continuation")
final <- rbind(C_clean, D_S)

#order <- final %>%
#  group_by(drug.treatment) %>%
#  summarize(NumPeople = n(), .groups = 'drop') %>%
#  as.data.frame() %>%
#  arrange(desc(NumPeople)) %>%
#  slice_head(n = 15)

group_counts <- final %>%
  mutate(drug.treatment = ifelse(!drug.treatment %in% order, "Other", drug.treatment)) %>%
  group_by(person_id, drug.treatment, FinalClassification.treatment) %>%
  ungroup() %>%
  group_by(drug.treatment, FinalClassification.treatment) %>%
  summarize(NumPeople = n(), .groups = 'drop') %>%
  arrange(drug.treatment) %>%
  as.data.frame()


group_counts$drug.treatment = factor(group_counts$drug.treatment, levels = c(order, "Other"))
group_counts$FinalClassification = factor(group_counts$FinalClassification, levels = c("Switching", "Discontinuation", "Continuation"))

colors <- c(
    "Discontinuation" = "#0D47A1FF",
    "Switching" = "#F57F17FF",
    "Continuation" = "#50c878")

p1e <- ggplot(group_counts, aes(x = drug.treatment, y = NumPeople, fill = FinalClassification)) +
  geom_bar(position = "stack", stat = "identity") +
  theme_classic() +
  scale_fill_manual(values = colors, name = "Outcome") +
  xlab("Common antidepressants") +
  ylab("AoU Participant Count") +
  labs(fill = "Antidepressant\nOutcome", title = "AoU: CIDI-noMDD") +
  theme(
    axis.text.x = element_text(angle = 60, hjust = 1, color = "black"),
    axis.title = element_text(face="bold"),
    legend.position = "none"
  ) +
  scale_y_continuous(
    breaks = seq(0, 50000, by = 200),
    labels = function(x) paste0(x / 1000, "K"))
p1e

In [ ]:
dat <- read_excel("/home/jupyter/workspaces/geneticsofantidepressantresponse/data/All_Results_PGS_LL.xlsx", sheet = "Table9")
dat_full <- dat %>%
    filter(Cohort == "Depression")

order <- dat_full %>%
    filter(DrugName.treatment != "Other") %>%
    group_by(DrugName.treatment) %>%
    summarise(Total = sum(NumPeople)) %>%
    arrange(desc(Total)) %>%
    pull(DrugName.treatment)

dat_full$FinalClassification.treatment = factor(dat_full$FinalClassification.treatment, levels = c("Switching", "Discontinuation", "Continuation"))
dat_full$DrugName.treatment = factor(dat_full$DrugName.treatment, levels = c(order, "Other"))


colors <- c(
    "Discontinuation" = "#0D47A1FF",
    "Switching" = "#F57F17FF",
    "Continuation" = "#50c878")

p1b_ll <- ggplot(dat_full, aes(x = DrugName.treatment, y = NumPeople, fill = FinalClassification.treatment)) +
  geom_bar(position = "stack", stat = "identity") +
  theme_classic() +
  scale_fill_manual(values = colors, name = "Outcome") +
  xlab("Common antidepressant") +
  ylab("LL Participant Count") +
  ggtitle("LL: Recorded Depression") +
  theme(
    axis.text.x = element_text(angle = 60, hjust = 1, color = "black"),
    axis.title = element_text(face="bold"),
    legend.position = "none"
  ) +
  scale_y_continuous(
    breaks = seq(0, 4500, by = 1000),
    labels = function(x) paste0(x / 1000, "K")) 
      
p1b_ll

In [ ]:
dat <- read_excel("/home/jupyter/workspaces/geneticsofantidepressantresponse/data/All_Results_PGS_LL.xlsx", sheet = "Table9")
dat_full <- dat %>%
    filter(Cohort == "No Depression")

#order <- dat_full %>%
#    filter(DrugName.treatment != "Other") %>%
#    group_by(DrugName.treatment) %>%
#    summarise(Total = sum(NumPeople)) %>%
#    arrange(desc(Total)) %>%
#    pull(DrugName.treatment)

dat_full$FinalClassification.treatment = factor(dat_full$FinalClassification.treatment, levels = c("Switching", "Discontinuation", "Continuation"))
dat_full$DrugName.treatment = factor(dat_full$DrugName.treatment, levels = c(order, "Other"))


colors <- c(
    "Discontinuation" = "#0D47A1FF",
    "Switching" = "#F57F17FF",
    "Continuation" = "#50c878")

p1c_ll <- ggplot(dat_full, aes(x = DrugName.treatment, y = NumPeople, fill = FinalClassification.treatment)) +
  geom_bar(position = "stack", stat = "identity") +
  theme_classic() +
  scale_fill_manual(values = colors, name = "Outcome") +
  xlab("Common antidepressant") +
  ylab("PharmLines Participant Count") +
  ggtitle("LL: No Recorded Depression") +
  theme(
    axis.text.x = element_text(angle = 60, hjust = 1, color = "black"),
    axis.title = element_text(face="bold"),
    legend.position = "none"
  ) +
  scale_y_continuous(
    breaks = seq(0, 4500, by = 1000),
    labels = function(x) paste0(x / 1000, "K")) 
      
p1c_ll

In [ ]:
topB <- plot_grid(p1b, p1c, ncol = 2, labels = c("A", ""))
middle <- plot_grid(p1d, p1e, ncol = 2)
bottom <- plot_grid(p1b_ll, p1c_ll, ncol = 2, labels = c("B", ""))
full <- plot_grid(topB, middle, bottom, ncol = 1)
full

# Save as PNG
ggsave(
  filename = "/home/jupyter/workspaces/geneticsofantidepressantresponse/data/Figures/Supp_Drug_Outcomes_Summary.png",
  plot     = full,
  width    = 10,
  height   = 12,
  dpi      = 600
)

In [ ]:
first_switch <- final %>%
  filter(FinalClassification.treatment == "Switching") %>%
  group_by(person_id) %>%
  filter(Incidence == min(Incidence)) %>%
  filter(FirstDispense.treatment.episode == min(FirstDispense.treatment.episode)) %>%
  filter(ExpectedEndDate.treatment.episode == max(ExpectedEndDate.treatment.episode)) %>%
  filter(ExpectedEndDate.treatment == min(ExpectedEndDate.treatment)) %>%
  ungroup() %>%
  mutate(Time2FirstSwitch = difftime(Incidence, ExpectedEndDate.treatment.episode, units = "days")) %>%
  mutate(TimeFirstDrug = difftime(ExpectedEndDate.treatment.episode, FirstDispense.treatment.episode, unit = "days")) %>%
  mutate(TimeSecondDrug = difftime(ExpectedEndDate.other.episode, FirstDispense.other.episode, unit = "days"))

cross_taper <- first_switch %>%
    filter(Time2FirstSwitch < 0)

switching_summary <- first_switch %>%
  summarise(
    Count = n(),
    
    # Time to first switch
    mean_t2s = mean(Time2FirstSwitch, na.rm = TRUE),
    median_t2s = median(Time2FirstSwitch, na.rm = TRUE),
    q1_t2s = quantile(Time2FirstSwitch, 0.25, na.rm = TRUE),
    q3_t2s = quantile(Time2FirstSwitch, 0.75, na.rm = TRUE),
    iqr_t2s = IQR(Time2FirstSwitch, na.rm = TRUE),
    
    # Time on first drug
    mean_tFD = mean(TimeFirstDrug, na.rm = TRUE),
    median_tFD = median(TimeFirstDrug, na.rm = TRUE),
    q1_tFD = quantile(TimeFirstDrug, 0.25, na.rm = TRUE),
    q3_tFD = quantile(TimeFirstDrug, 0.75, na.rm = TRUE),
    iqr_tFD = IQR(TimeFirstDrug, na.rm = TRUE),
    
    # Time on second drug
    mean_tSD = mean(TimeSecondDrug, na.rm = TRUE),
    median_tSD = median(TimeSecondDrug, na.rm = TRUE),
    q1_tSD = quantile(TimeSecondDrug, 0.25, na.rm = TRUE),
    q3_tSD = quantile(TimeSecondDrug, 0.75, na.rm = TRUE),
    iqr_tSD = IQR(TimeSecondDrug, na.rm = TRUE)
  )

switching_summary

In [ ]:
switch_df <- first_switch %>%
  mutate(first_class = ifelse(!Category.treatment %in% c("SSRI", "SNRI", "TCA", "NaSSA", "SARI", "NDRI"), "Other", Category.treatment)) %>%
  mutate(second_class = ifelse(!Category.other %in% c("SSRI", "SNRI", "TCA", "NaSSA", "SARI", "NDRI"), "Other", Category.other)) %>%
  group_by(first_class, second_class) %>%
  summarise(
    freq = n(),
    .groups = "drop"
  )

In [ ]:
p2 <- ggplot(
  switch_df,
  aes(
    axis1 = first_class,
    axis2 = second_class,
    y = freq
  )
) +
  geom_alluvium(aes(fill = first_class),
                width = 0.15,
                alpha = 0.8) +
  geom_stratum(width = 0.15) +
  geom_text(
    stat = "stratum",
    aes(label = after_stat(stratum)),
    size = 3,
    fontface = "bold"
  ) +
  scale_x_discrete(
    limits = c("First\ndrugclass", "Second\ndrugclass"),
    expand = c(0.1, 0.1)
  ) +
  labs(
    x = NULL,
    y = "AoU Participant Count (N=27,389)",
    fill = "First class",
    title = NULL
  ) +
  theme_classic() +
  theme(
    legend.position = "none",
    panel.grid.major = element_blank(),
    panel.grid.minor = element_blank(),
    legend.title = element_text(face = "bold"),
    axis.title = element_text(face = "bold"),
    axis.text.x = element_text(size = 12, face = "bold", color = "black")
  )
p2

In [ ]:
#-- ggalluvial plot for LL
library(readxl)
dat <- read_excel("/home/jupyter/workspaces/geneticsofantidepressantresponse/data/All_Results_PGS_LL.xlsx", sheet = "Table7")
sum(dat$freq)

p2b <- ggplot(
  dat,
  aes(
    axis1 = first_class,
    axis2 = second_class,
    y = freq
  )
) +
  geom_alluvium(aes(fill = first_class),
                width = 0.15,
                alpha = 0.8) +
  geom_stratum(width = 0.15) +
  geom_text(
    stat = "stratum",
    aes(label = after_stat(stratum)),
    size = 3,
    fontface = "bold"
  ) +
  scale_x_discrete(
    limits = c("First\ndrugclass", "Second\ndrugclass"),
    expand = c(0.1, 0.1)
  ) +
  labs(
    x = NULL,
    y = "Pharmlines Participant Count (N=2,192)",
    fill = "First class",
    title = NULL
  ) +
  theme_classic() +
  theme(
    legend.position = "none",
    #axis.text.y = element_blank(),
    panel.grid.major = element_blank(),
    panel.grid.minor = element_blank(),
    legend.title = element_text(face = "bold"),
    axis.title = element_text(face = "bold"),
    axis.text.x = element_text(size = 12, face = "bold", color = "black")
  ) +
  scale_fill_manual(
    values = c(
      "SSRI" = "#A58AFF",
      "TCA" = "#FB61D7",
      "SNRI" = "#00B6EB",
      "NaSSA" = "#F87660",
      "Other" = "#53B400"
    )
  )
p2b

In [ ]:
bottom <- plot_grid(p2, p2b, nrow = 1, labels = c("C", "D"))
all <- plot_grid(top, bottom, nrow =2)
all

# Save as PNG
ggsave(
  filename = "/home/jupyter/workspaces/geneticsofantidepressantresponse/data/Figures/Main_Class_Switch_Summary.png",
  plot     = all,
  width    = 10,
  height   = 10,
  dpi      = 600
)

In [ ]:
#-- Find distribution of age at first dispense
age <- both %>%
    select(person_id, age_at_first_AD, date_of_birth) %>%
    distinct()

# Calculate stats for annotation
mean_age <- mean(age$age_at_first_AD, na.rm = TRUE)
sd_age <- sd(age$age_at_first_AD, na.rm = TRUE)
mean_age
sd_age

p6 <- ggplot(age, aes(x = age_at_first_AD)) +
  geom_density(fill = "#4A90D9", colour = "#2C5F8A", alpha = 0.4, linewidth = 0.8) +
  geom_vline(xintercept = 48.7, linetype = "dashed", colour = "#2C5F8A", linewidth = 0.8) +
  annotate("text", x = 49.7, y = 0.0245, 
           label = "Mean = 48.7 (SD = 15.5)", 
           hjust = 0, colour = "#2C5F8A", size = 4) +
  scale_x_continuous(limits = c(15, 95), breaks = seq(20, 90, 20)) +
  labs(x = "Age at first recorded antidepressant prescription (years)", y = "Density") +
  theme_minimal() +
  theme(
    panel.grid.minor = element_blank(),
    panel.grid.major.x = element_blank(),
    axis.line = element_line(colour = "grey50")
  )
p6

# Save as PNG
ggsave(
  filename = "/home/jupyter/workspaces/geneticsofantidepressantresponse/data/Figures/Supp_Age_First_AD_Dispense.png",
  plot     = p6,
  width    = 8,
  height   = 6,
  dpi      = 600
)


In [ ]:
install.packages("hexbin")

In [ ]:
library(hexbin)

#-- YOB regressed on date at first prescription
age <- age %>%
    mutate(year_of_birth = year(ymd(as.Date(date_of_birth))))
yob <- ggplot(age, aes(x = year_of_birth, y = age_at_first_AD)) +
  geom_hex(binwidth = c(5, 5)) +
  scale_fill_gradient(
    low = "#D9E8F5",
    high = "#1F4E79",
    na.value = "grey92",
    name = "N",
    limits = c(20, NA),
    oob = scales::squish      # values < 20 are shown as NA -> grey
  ) +
  scale_y_continuous(limits = c(15, 95), breaks = seq(20, 90, 20)) +
  labs(
    y = "Age at first recorded antidepressant record (years)",
    x = "Year of birth"
  ) +
  theme_minimal() +
  theme(
    panel.grid.minor = element_blank(),
    panel.grid.major.x = element_blank(),
    axis.line = element_line(colour = "grey50"),
    legend.position = "right"
  )
yob

# Save as PNG
ggsave(
  filename = "/home/jupyter/workspaces/geneticsofantidepressantresponse/data/Figures/Supp_Age_YOB_on_First_AD_Dispense.png",
  plot     = yob,
  width    = 8,
  height   = 6,
  dpi      = 600
)

In [ ]:
sex <- both %>%
    select(person_id, sex_at_birth) %>%
    distinct()
table(sex$sex_at_birth, useNA = "always")

In [ ]:
#-- Read in predicted ancestries
workspace_bucket <- Sys.getenv("WORKSPACE_BUCKET")
anc_file_path <- file.path(workspace_bucket, "ancestry/ancestry_preds.tsv")
anc_df <- read_tsv(pipe(sprintf("gsutil cat %s", anc_file_path)))
anc_df <- anc_df %>% select(research_id, ancestry_pred)
ancestries <- unique(anc_df$ancestry_pred)

dat <- both %>%
    left_join(anc_df, by = c("person_id" = "research_id"))
dat <- dat %>%
          mutate(
            ancestry_pred = factor(ancestry_pred)
      )

In [ ]:
anc <- dat %>%
    select(person_id, ancestry_pred) %>%
    distinct()
table(anc$ancestry_pred, useNA = "always")

In [ ]:
dim(dat)
length(unique(dat$person_id))

In [ ]:
mdd <- both %>%
    select(person_id, Depression) %>%
    distinct()
table(mdd$Depression, useNA = "always")

In [ ]:
cidi_mdd <- both %>%
    select(person_id, Depression, MDD_Incl) %>%
    distinct()
table(cidi_mdd$MDD_Incl, useNA = "always")


In [ ]:
#-- depression cohort with CIDI-MDD
cidi_mdd_dep <- both %>%
    select(person_id, Depression, MDD_Incl) %>%
    filter(Depression == 1) %>%
    distinct()
dim(cidi_mdd_dep)
table(cidi_mdd_dep$MDD_Incl, useNA = "always")
7038